In [ ]:
import torch
import torch.nn as nn
import math

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import time
import matplotlib.pyplot as plt

from tqdm import tqdm 

In [ ]:
def convert_col_into_float(df, list_cols):
    for col in list_cols: 
        df[col] = df[col].astype(str)
        df[col] = df[col].str.replace(',', '.')
        df[col] = df[col].astype(np.float32)
    return df

df = pd.read_csv("file_name", index_col = 'time', parse_dates = ['time'])

df.head()
list_cols = list(df.columns)
print("dataset variables", list_cols)
df = conver_col_into_float(df, list_cols)
data = df.values

In [ ]:
# split raw data into sequences 

def split_dataset_into_seq(dataset, start_index = 0, end_index = None, history_size = 13, step = 1):
    data = []
    start_index = start_index + history_size
    if end_index is None:
        end_index = len(dataset)
    for i in range (start_index, end_index):
        indices = range(i - history_size, i, step)
        data.append(dataset[indices])
    return np.array(data)   # 3 chiều: 1 list, a số hàng, b số cột

In [ ]:
# split sequence data (from raw data) into train and val 
def split_dataset(data, TRAIN_SPLIT = 0.7, VAL_SPLIT = 0.5, save_path = None):
    
    data_mean = data.mean(axis = 0)
    data_std = data.std(axis = 0)
    data = (data - data_mean) / data_std  # Standardiztion 
    
    data_in_seq = split_dataset_into_seq(data, start_index = 0, end_index = None, history_size = 13, step = 1)
    train_data, val_data = train_test_split(data_in_seq, train_size = TRAIN_SPLIT, shuffle = False, random_state = 123)
    val_data, test_data = train_test_split(val_data, train_size = VAL_SPLIT, shuffle = False, random_state = 123)
    
    return train_data, val_data, test_data 

In [ ]:
# split each sequence data (train, val, test) to input past and target future - 1 step
def split_fn(chunk): 
    inputs = torch.tensor(chunk[:, :-1, :], device = device)
    targets = torch.tensor(chunk[]:, 1:, :, device = device)
    return inputs, targets 

In [ ]:
# finalize train_loader, val_loader, test_loader 

def data_to_dataset(train_data, val_data, test_data, batch_size = 32): 
    x_train, y_train = split_fn(train_data)
    x_val, y_val = split_fn(val_data)
    x_test, y_test = split_fn(test_data)
    
    train_dataset = torch.utils.data.TensorDataset(x_train, y_train)
    val_dataset = torch.utils.data.TensorDataset(x_val, y_val)
    test_dataset - torch.utils.data.TensorDataset(x_test, y_test)
    
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size = batch_size)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size = batch_size)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size = batch_size)
    return train_loader, val_loader, test_loader

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
train_data, val_data, test_data = split_dataset(data)
train_dataset, val_dataset, test_dataset = data_to_dataset(train_data, val_data, test_data)

Implementation of the Transformer model

In [ ]:
class MultiHeadAttention(nn.Module): 
    super(MultiHeadAttention, self).__init__()
    self.H = H
    self.D = D
    
    self.wq = nn.Linear(D, D*H)
    self.wk = nn.Linear(D, D*H)
    self.wv = nn.Linear(D, D*H)
    
    self.dense = nn.Linear(D*H, D)
    def concat_heads(self, x):
        B, H, S, D = x.shape
        x = x.permute((0,2,1,3)).contiguous()
        x = x.reshape((B, S, H*D))
        return x
        
    def split_heads(self, x):
        B, S, D_H = x.shape
        x = x.reshape(B, S, self.H, self.D)
        x = x.permute((0,2,1,3))
        return x
        
    def forward(self, x, mask):
        q = self.wq(x)
        k = self.wk(x)
        v = self.wv(x)
        
        q = self.split_heads(q)
        k = self.split_heads(k)
        v = self.split_heads(v)
        
        attention_scores = torch.matmul(q, k.transpose(-1, -2))
        attention_scores = attention_scores / math.sqrt(self.D)
        
        if mask is not None:
            attention_scores += (mask * -1e9)
            
        attention_weights = nn.Softmax(dim = -1)(attention_scores)
        scaled_attention = torch.matmul(attention_weights, v)
        concat_attention = self.concat_heads(scaled_attention)
        output = self.dense(concat_attention)
        
        return output, attention_weights 

In [ ]:
B, S, H, D = 9, 11, 5, 8
mha = MultiHeadAttention(D, H)
out, att = mha.forward(torch.zeros(B, S, D), mask = None)
out.shape, att.shape

In [ ]:
# Positional encodings
def get_angles(pos, i, D):
    angle_rates = 1 / np.power(10000, (2 * (i // 2)) / np.float32(D))
    return pos * angle_rates


def positional_encoding(D, position=20, dim=3, device=device):
    angle_rads = get_angles(np.arange(position)[:, np.newaxis],
                            np.arange(D)[np.newaxis, :],
                            D)
    # apply sin to even indices in the array; 2i
    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
    # apply cos to odd indices in the array; 2i+1
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])
    if dim == 3:
        pos_encoding = angle_rads[np.newaxis, ...]
    elif dim == 4:
        pos_encoding = angle_rads[np.newaxis,np.newaxis,  ...]
    return torch.tensor(pos_encoding, device=device)

In [ ]:
# function that implement the look_ahead mask for masking future time steps.
def create_look_ahead_mask(size, device=device):
    mask = torch.ones((size, size), device=device)
    mask = torch.triu(mask, diagonal=1)
    return mask  # (size, size)

In [ ]:
create_look_ahead_mask(6)

In [ ]:
class TransformerLayer(nn.Module):
    def __init__(self, D, H, hiddne_mlp_dim, dropout_rate):
        super(TransformerLayer, self).__init__()
        self.dropout_rate = dropout_rate
        self.mlp_hidden = nn.Linear(D, hidden_mlp_dim)
        self.mlp_out = nn.Linear(hidden_mlp_dim, D)
        self.layernorm1 = nn.LayerNorm(D, eps = 1e-9)
        self.layernorm2 = nn.LayerNorm(D, eps = 1e-9)
        self.dropout1 = nn.Dropout(dropout_rate)
        self.dropout2 = nn.Dropout(dropout_rate)
        
        self.mha = MultiHeadAttention(D, H)
        
    def forward(self, x, look_ahead_mask):
        attn, attn_weights = self.mha(x, look_ahead_mask)
        attn = self.dropout1(attn)
        attn = self.layernorm1(attn + x)
            
        mlp_act = torch.relu(self.mlp_hidden(attn))
        mlp_act = self.mlp_out(mlp_act)
        mlp_act = self.dropout2(mlp_act)
            
        output = self.layernorm2(mlp_act + attn)
            
        return output, attn_weights

In [ ]:
dl = TransformerLayer(16,3,32, 0.1)
out, attn = dl(x=torch.zeros(5,7,16), look_ahead_mask = None)
out.shape, attn.shape

In [ ]:
class Transformer(nn.Module):
    def __init__ (self, num_layers, D, H, hidden_mlp_dim, inp_features, out_features, dropout_rate):
        super(Transformer, self).__init__()
        self.sqrt_D = torch.tensor(math.sqrt(D))
        self.num_layers = num_layers
        self.input_projection = nn.Linear(inp_features, D)
        self.output_projection = nn.Linear(D, out_features)
        self.pos_encoding = positional_encoding(D)
        self.dec_layers = nn.ModuleList([TransformerLaer(D, H, hidden_mlp_dim, dropout_rate = dropout_rate)
                                        for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout_rate)
        
    def forward(self, x, mask):
        B, S, D = x.shape
        attention_weights = {}
        x = self.input_projection(x)
        x *= self.sqrt_D
        x += self.pos_encoding[:, :S, :]
        x = self.dropout(x)
        for i in range(self.num_layers):
            x, block = self.dec_layers[i](x = x, look_ahead_mask = mask)
            attention_weights['decoder_layer{}'.format(i+1)] = block
            
        x = self.output_projection(x)
        
        return x, attention_weights 

In [ ]:
transformer = Transformer(num_layers=1, D=32, H=1, hidden_mlp_dim=32,
                                       inp_features=28, out_features=20, dropout_rate=0.1)

transformer.to(device)
(inputs, targets) = next(iter(train_dataset))

S = inputs.shape[1]
mask = create_look_ahead_mask(S)
out, attn = transformer (x=inputs, mask=mask)
out.shape, attn["decoder_layer1"].shape

Training the Transformer

In [ ]:
param_sizes = [p.numel() for p in transformer.parameters()]

print(f"number of weight/biases matrices: {len(param_sizes)}"
     f"for a total of {np.sum(param_sizes)} parameters")

In [ ]:
transformer = Transformer(num_layers=1, D=32, H=4, hidden_mlp_dim=32,
                          inp_features=28, out_features=20, dropout_rate=0.1).to(device)
optimizer = torch.optim.RMSprop(transformer.parameters(),
                                lr=0.00005)

In [ ]:
from tqdm import tqdm 

n_epochs = 20
niter = len(train_dataset)
losses, val_losses = [], []

for e in tqdm(range(n_epochs)):
    transformer.train()
    sum_train_loss = 0.0
    for x, y in train_dataset:
        S = x.shape[1]
        mask = create_look_ahead_mask(S)
        out, _ = transformer(x, mask)
        loss = torch.nn.MSELoss()(out, y)
        sum_train_loss += loss.item()
        loss.backward()
        optimizer.step()
    losses.append(sum_train_loss / niter)
    
    transformer.eval()
    sum_val_loss = 0.0
    for i, (x,y) in enumerate(val_dataset):
        S = x.shape[1]
        mask = create_look_ahead_mask(S)
        out, _ = transformer(x, mask)
        loss = torch.nn.MSELoss()(out, y)
        sum_val_loss += loss.item()
    val_losses.append(sum_val_loss / (i + 1))

In [ ]:
plt.plot(losses)
plt.plot(val_losses)